In [1]:
# File Authorship Information
__author__ = """Francesca Pelusi, Matteo Lulli, Christophe Coreixas, Mauro Sbragaglia, Xiaowen Shan"""
__copyright__ = """Copyright 2025, Francesca Pelusi, Matteo Lulli, Christophe Coreixas, Mauro Sbragaglia, Xiaowen Shan, idea.deploy"""
__license__ = """Permission is hereby granted, free of charge, 
to any person obtaining a copy of this software and associated 
documentation files (the "Software"), to deal in the Software 
without restriction, including without limitation the rights to 
use, copy, modify, merge, publish, distribute, sublicense, 
and/or sell copies of the Software, 
and to permit persons to whom the Software is furnished to do so, 
subject to the following conditions:
The above copyright notice and this permission notice shall be 
included in all copies or substantial portions of the Software.
THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, 
EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES 
OF MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND 
NONINFRINGEMENT. IN NO EVENT SHALL THE AUTHORS OR COPYRIGHT 
HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY, 
WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER 
DEALINGS IN THE SOFTWARE."""
__maintainer__ = "Francesca Pelusi, Matteo Lulli"
__email__ = "pelusi.fra@gmail.com, matteo.lulli@gmail.com"
__status__ = "Development"

In [2]:
# Development cell
%load_ext autoreload
%autoreload 2

# A note on the lattice momentum balance in the lattice Boltzmann interaction-framework

Authors: Francesca Pelusi, Matteo Lulli, Christophe Coreixas, Mauro Sbragaglia, Xiaowen Shan


**Abstract:**
In this note, we show how the exploitation of the lattice momentum balance condition allows to envisage an analytical procedure to define the lattice pressure tensor (LPT) for the multi-phase Shan-Chen (SC) lattice Boltzmann method (LBM) with single-range potential. This construction ensures that the LPT normal component to a flat interface is constant to machine precision on each lattice node, i.e., it exactly implements the mechanical equilibrium condition on the lattice. We demonstrate the robustness of the approach by providing analytical expressions for the coexistence curves for different choices of the pseudo-potential and forcing schemes in the SC-LBM. This paper offers a novel, rigorous perspective for controlling the LPT in the SC-LBM, paving the way for its application in more general settings.

# Reproducibility

This document is intended for those interested readers who want to reproduce the results reported in the paper [https://arxiv.org/abs/2503.05743](https://arxiv.org/abs/2503.05743). In the present case the computational resources needed should be available in general. 

Next development steps will include a class to measure the time required by each cell and output it in a .json file which can be sent to [matteo.lulli@gmail.com](mailto:matteo.lulli@gmail.com) and [pelusi.fra@gmail.com](mailto:pelusi.fra@gmail.com) so that average execution times will be available and organized according to the hardware.

Each subsection can be executed independently and reproduce the results which will be stored locally in the directory 'reproduced-data', so that the data will be generated only once.

Plots can also be generated using the same scripts employed for the figures of the paper. Since there are some issues in executing these scripts in the Jupyter environment we include them as separated files which are called from the cells themselves. **In order to reproduce the plots a working 'latex' installation is necessary to be present on the system.**

This file will be kept updated for new local features and developments in the parent project [**idea.deploy**](https://github.com/lullimat/idea.deploy)

This file is supposed to be pulled from the repository [https://arxiv.org/abs/2503.05743](https://arxiv.org/abs/2503.05743), from within the "papers" directory the idea.deploy project.

In [3]:
# Importing Modules
import sys
sys.path.append("../../")

from idpy.IdpyCode import CUDA_T, OCL_T, CTYPES_T, idpy_langs_sys

from idpy.LBM.SCFStencils import SCFStencils, BasisVectors
from idpy.LBM.LBM import XIStencils, ShanChenMultiPhase, CheckUConvergence

from LBM_proxy import PShanChenMultiPhase, GetSnapshotN
# from SCThermo_proxy import ShanChanEquilibriumCache, ShanChen
from idpy.LBM.SCThermo import ShanChanEquilibriumCache, ShanChen

import sympy as sp
import numpy as np
from scipy import interpolate
from collections import defaultdict

In [4]:
# Defining Symbols and Stencil D2E4
n = sp.symbols('n')

_D2E4_P4F4 = SCFStencils(E = BasisVectors(x_max = 1), 
                         len_2s = [1, 2])
_D2E4_P4F4.FindWeights()

import time
from collections import defaultdict

'''
lattice and relaxation details
'''
_tau, _c_s2 = 1, 1/3

# Defining the pseudopotentials, psi_codes, G and finding flat interface equilibrium values
psis = [sp.exp(-1/n), 1 - sp.exp(-n)]

psi_codes = {psis[0]: 'exp((NType)(-1./ln))', 
             psis[1]: '1. - exp(-(NType)ln)',}

_Gc = {psis[0]: -2.46301869964355, 
       psis[1]: -1.3333333333333333}

_psi_sym = psis[0]
_G_list = {psis[0]: -np.exp(np.linspace(np.log(2.47), np.log(6), 2 ** 5)), 
           psis[1]: -np.exp(np.linspace(np.log(_Gc[psis[1]] * 2.47 / _Gc[psis[0]]), 
                                        np.log(_Gc[psis[1]] * 6 / _Gc[psis[0]]), 2 ** 5))}



_G_list_sims = \
    {    
        'guo': {0.8: {psis[0]: _G_list[psis[0]], psis[1]: _G_list[psis[1]][:-15]}, 
                1: {psis[0]: _G_list[psis[0]], psis[1]: _G_list[psis[1]][:-15]}, 
                1.2: {psis[0]: _G_list[psis[0]], psis[1]: _G_list[psis[1]][:-15]}},
    
        'sc': {0.8: {psis[0]: _G_list[psis[0]], psis[1]: _G_list[psis[1]][:-11]}, 
               1: {psis[0]: _G_list[psis[0]], psis[1]: _G_list[psis[1]]}, 
               1.2: {psis[0]: _G_list[psis[0]], psis[1]: _G_list[psis[1]]}},
    
        'ks': {0.8: {psis[0]: _G_list[psis[0]], psis[1]: _G_list[psis[1]]}, 
               1: {psis[0]: _G_list[psis[0]], psis[1]: _G_list[psis[1]]}, 
               1.2: {psis[0]: _G_list[psis[0]], psis[1]: _G_list[psis[1]]}},
    }

The next cells will generate the values for the equilibrium properties, i.e. densities, pressure and surface tension.

If one wishes to skip this step, which at the moment is time consuming, one can run the command

`cp SCEqCache-non-reproduced.json SCEqCache.json`

in the cell below and the pass to the following one for loading the values in the different structures

In [5]:
# ! cp SCEqCache-non-reproduced.json SCEqCache.json

In [6]:
# Generate Kupershtokh forcing equilibrium densities and surface tension

_th_data = defaultdict( # Forcing type
    lambda : defaultdict( # _tau
        lambda : defaultdict ( # _psi
            lambda : defaultdict (dict) # _G
        )
    )
)

from idpy.Utils.SimpleTiming import SimpleTiming

def ForcingName(forcing):
    if forcing == 'sc':
        return 'Shan-Chen'
    if forcing == 'ks':
        return 'Kupershtokh'
    if forcing == 'guo':
        return 'Guo'



for _forcing in ['guo', 'sc', 'ks']:
    
    _st = SimpleTiming()
    _st.Start()

    print("START: ", ForcingName(_forcing), "Forcing")
    for _tau in [0.8, 1, 1.2]:
        for _psi in psis:
            for _G in _G_list_sims[_forcing][_tau][_psi]:
                print("Forcing:", ForcingName(_forcing), "tau:", _tau,
                      "G:", _G, "Psi:", _psi)
                # _forcing_cache = _forcing if _forcing != 'mod' else 'guo'
                _forcing_cache = _forcing
                sc_eq_cache = \
                    ShanChanEquilibriumCache(
                        stencil = _D2E4_P4F4, 
                        psi_f = _psi, 
                        G = _G,
                        c2 = _c_s2, 
                        n_eps = 1e-6,
                        forcing = _forcing_cache, 
                        tau = _tau
                    )

                _eq_params = sc_eq_cache.GetFromCache()
                _th_data[_forcing][_tau][_psi][_G] = _eq_params
                print(_eq_params)
                print("Density ratio:", _eq_params['n_l']/_eq_params['n_g'])
                print()
                print("-----------------------------------------------------")
                print()
                print()
    
    print("END: ", ForcingName(_forcing), "Forcing")

    _st.End()
    _st.PrintElapsedTime()
    print()
    print()

# Organizing the equilibrium data in a dictionary as a function of G
_th_obs_data = defaultdict( # Forcing type
    lambda : defaultdict( # tau
        lambda : defaultdict( # psi
            lambda : defaultdict (dict) # obs
        )
    )
)

for _forcing in ['guo', 'sc', 'ks']:
    print(ForcingName(_forcing), "Forcing")
    for _tau in [0.8, 1, 1.2]:
        for _psi in psis:
            for _obs in ['n_g', 'n_l', 'p_0', 'sigma_f']:
                _th_obs_data[_forcing][_psi][_tau][_obs] = []

                for _G in _G_list_sims[_forcing][_tau][_psi]:
                    _th_obs_data[_forcing][_psi][_tau][_obs] += \
                        [_th_data[_forcing][_tau][_psi][_G][_obs]]

                _th_obs_data[_forcing][_psi][_tau][_obs] = \
                    np.array(_th_obs_data[_forcing][_psi][_tau][_obs])

                _dummy_G = _G_list_sims[_forcing][_tau][_psi][0]
                
                _th_obs_data[_forcing][_psi][_tau]['n_c'] = \
                    _th_data[_forcing][_tau][_psi][_dummy_G]['n_c']
                _th_obs_data[_forcing][_psi][_tau]['G_c'] = \
                    _th_data[_forcing][_tau][_psi][_dummy_G]['G_c']
                _th_obs_data[_forcing][_psi][_tau]['G_list'] = \
                    _G_list_sims[_forcing][_tau][_psi]

## Simulations

### Hardware selection

In [7]:
## Displaying the available Hardware and target languages
from idpy.IdpyCode import IdpyHardware
IdpyHardware()

- Given that the simulations are all quasi-one-dimensional, i.e., the domain size is relatively small, it is actually convenient to use the CPU through CTYPES
- The cell below is already set with this choice in mind; the total simulations runtime is roughly 1.5 hrs on a M1 Max CPU
- In case one wishes to use another execution environment, according to the installation, one can set

    `_lang = OCL_T` or `_lang = CUDA_T` with `_kind = 'cpu'` or `_kind = 'gpu'` accordingly

In [8]:
# Setting the language and the device

_lang, _device, _kind = CUDA_T, 0, 'gpu'

### Simulations Parameters

In [9]:
# Defining the pseudopotentials, psi_codes, G
psis = [sp.exp(-1/n), 1 - sp.exp(-n)]

psi_codes = {psis[0]: 'exp((NType)(-1./ln))', 
             psis[1]: '1. - exp(-(NType)ln)',}

_Gc = {psis[0]: -2.46301869964355, 
       psis[1]: -1.3333333333333333}

_nc = {psis[0]: 1.0, psis[1]: 0.6931471805599453}

_Pc = {psis[0]: _nc[psis[0]] / 3 + 0.5 * _Gc[psis[0]] * (psis[0].subs(n, _nc[psis[0]]).evalf() ** 2), 
       psis[1]: _nc[psis[1]] / 3 + 0.5 * _Gc[psis[1]] * (psis[1].subs(n, _nc[psis[1]]).evalf() ** 2)}


'''
For Christophe:
in order to test higher order equilibrium/forcings with reasonable lattice sizes just
copy the function SimulationsParams and change all the values that are not 256 to 256
like:
int(121 * (_Gc[_psi]/G)) -> int(256 * (_Gc[_psi]/G))
'''
def SimulationsParams(psi_sym, forcing, G, tau):
    _LX, _LY, _psi_code = 0, 0, ''
        
    if _psi == psis[0]:
        if (forcing == 'guo') and G > -3.583761320830255:
            _LX, _LY, _psi_code = int(256 * (_Gc[_psi]/_G)), 5, psi_codes[_psi]
        if (forcing == 'guo') and G <= -3.583761320830255:
            _LX, _LY, _psi_code = int(121 * (_Gc[_psi]/G)), 11, psi_codes[_psi]                            
        if (forcing == 'guo') and G <= -3.3843171808360184:
            _LX, _LY, _psi_code = int(111 * (_Gc[_psi]/G)), 11, psi_codes[_psi]                
        if (forcing == 'guo') and G <= -4.018603111761019:
            _LX, _LY, _psi_code = int(101 * (_Gc[_psi]/G)), 11, psi_codes[_psi]
        if (forcing == 'guo') and G <= -4.135320037598548:
            return False, False, False

        if _forcing == 'sc' or forcing == 'ks':
            _LX, _LY, _psi_code = int(256 * (_Gc[_psi]/G)), 5, psi_codes[_psi]

        if (forcing == 'sc' and tau == 0.8) and G > -3.3843171808360184:
            _LX, _LY, _psi_code = int(256 * (_Gc[_psi]/G)), 11, psi_codes[_psi]
        if (forcing == 'sc' and tau == 0.8) and G <= -3.3843171808360184:
            _LX, _LY, _psi_code = int(121 * (_Gc[_psi]/G)), 11, psi_codes[_psi]
        if (forcing == 'sc' and tau == 0.8) and G <= -4.018603111761019:
            _LX, _LY, _psi_code = int(101 * (_Gc[_psi]/G)), 11, psi_codes[_psi]
        if forcing == 'sc' and tau == 0.8 and G <= -5.0529755042462305:
            _LX, _LY, _psi_code = int(91 * (_Gc[_psi]/G)), 11, psi_codes[_psi]
        if forcing == 'sc' and tau == 0.8 and G <= -5.199734900679765:
            return False, False, False
        
        if forcing == 'sc' and tau == 1.2 and G <= -5.830653601496859:
            _LX, _LY, _psi_code = int(121 * (_Gc[_psi]/G)), 11, psi_codes[_psi]
        if forcing == 'sc' and tau == 1.2 and G <= -6:
            return False, False, False

        if forcing == 'ks' and tau == 0.8 and G <= -3.288796859841539:
            _LX, _LY, _psi_code = int(121 * (_Gc[_psi]/G)), 11, psi_codes[_psi]

    if _psi == psis[1]:                    
        if (forcing == 'guo') and G > -1.8320700969239085:
            _LX, _LY, _psi_code = int(256 * (_Gc[_psi]/G)), 5, psi_codes[_psi]
        if (forcing == 'guo') and G <= -1.8320700969239085:
            _LX, _LY, _psi_code = int(128 * (_Gc[_psi]/G)), 5, psi_codes[_psi]
        if (forcing == 'guo') and G <= -1.9400374136279197:
            return False, False, False

        ## LX_175_G_-1.9400374136279197_tau_1_F_sc_psi_1.hdf5
        if forcing == 'sc' and _tau == 0.8 and G > -1.996384158163539:
            _LX, _LY, _psi_code = int(256 * (_Gc[_psi]/G)), 11, psi_codes[_psi]
        if forcing == 'sc' and (_tau == 1 or _tau == 1.2) and G > -2.0543674461995365:
            _LX, _LY, _psi_code = int(256 * (_Gc[_psi]/G)), 11, psi_codes[_psi]
            
        if forcing == 'sc' and tau == 0.8 and _G <= -1.996384158163539:
            _LX, _LY, _psi_code = int(121 * (_Gc[_psi]/G)), 11, psi_codes[_psi]
        if forcing == 'sc' and tau == 1 and _G <= -2.0543674461995365:
            _LX, _LY, _psi_code = int(121 * (_Gc[_psi]/G)), 11, psi_codes[_psi]            
        if forcing == 'sc' and tau == 1.2 and _G <= -2.0543674461995365:
            _LX, _LY, _psi_code = int(121 * (_Gc[_psi]/G)), 11, psi_codes[_psi]
        if forcing == 'sc' and G <= -2.114034809756629:
            return False, False, False

        if forcing == 'ks' and G > -1.996384158163539:
            _LX, _LY, _psi_code = int(256 * (_Gc[_psi]/G)), 11, psi_codes[_psi]
        if forcing == 'ks' and G <= -1.996384158163539:
            _LX, _LY, _psi_code = int(121 * (_Gc[_psi]/G)), 11, psi_codes[_psi]
        if forcing == 'ks' and G <= -2.0543674461995365:
            _LX, _LY, _psi_code = int(101 * (_Gc[_psi]/G)), 11, psi_codes[_psi]
        if forcing == 'ks' and G <= -2.114034809756629:
            return False, False, False
        
    return _LX, _LY, _psi_code
    
def SimulationsAnalysisGLimit(psi_sym, forcing, G, tau):
    if _psi == psis[0]:
        if (forcing == 'guo') and G <= -4.135320037598548:
            return False
        if forcing == 'sc' and tau == 0.8 and G <= -5.199734900679765:
            return False                       
        if forcing == 'sc' and tau == 1.2 and G <= -6:
            return False

    if _psi == psis[1]:
        if (forcing == 'guo') and G <= -1.9400374136279197:
            return False
        if forcing == 'sc' and G <= -2.114034809756629:
            return False
        if forcing == 'ks' and G <= -2.114034809756629:
            return False
        
    return True

### Simulations Runs

In [10]:
# Folding Comment
from idpy.Utils.SimpleTiming import SimpleTiming
from idpy.Utils.ManageData import ManageData

def TotalNormalPSC(_n, _psi, _psi_p1, _psi_m1, _G, _tau, _c_s2):
    _tot_p_n = \
        _n * _c_s2 + \
        _G * _psi * (_psi_p1 + _psi_m1) / 4 + \
        ((_tau - 1/2) ** 2) * (_G ** 2) * (_psi ** 2) * ((_psi_p1 - _psi_m1) ** 2) / 12 / _n / _c_s2
    return _tot_p_n

def TotalTangentialPSC(_n, _psi, _psi_p1, _psi_m1, _G, _tau, _c_s2):
    _tot_t_n = \
        _n * _c_s2 + \
        _G * _psi * ((1 - 1 / 3) * _psi + (_psi_p1 + _psi_m1) / 2 / 3) / 2 + \
        ((_tau - 1/2) ** 2) * (_G ** 2) * (_psi ** 2) * ((_psi_p1 - _psi_m1) ** 2) / 6 / 3 / _n / _c_s2
    return _tot_t_n

def TotalNormalPGuo(_n, _psi, _psi_p1, _psi_m1, _G, _tau, _c_s2):
    _tot_p_n = \
        _n * _c_s2 + \
        _G * _psi * (_psi_p1 + _psi_m1) / 4
    return _tot_p_n

def TotalTangentialPGuo(_n, _psi, _psi_p1, _psi_m1, _G, _tau, _c_s2):
    _tot_t_n = \
        _n * _c_s2 + \
        _G * _psi * ((1 - 1 / 3) * _psi + (_psi_p1 + _psi_m1) / 2 / 3) / 2
    return _tot_t_n

def TotalNormalPKS(_n, _psi, _psi_p1, _psi_m1, _G, _tau, _c_s2):
    _tot_p_n = \
        _n * _c_s2 + \
        _G * _psi * (_psi_p1 + _psi_m1) / 4 + \
        (_G ** 2) * (_psi ** 2) * ((_psi_p1 - _psi_m1) ** 2) / 48 / _n / _c_s2
    return _tot_p_n

def TotalTangentialPKS(_n, _psi, _psi_p1, _psi_m1, _G, _tau, _c_s2):
    _tot_t_n = \
        _n * _c_s2 + \
        _G * _psi * ((1 - 1 / 3) * _psi + (_psi_p1 + _psi_m1) / 2 / 3) / 2 + \
        (_G ** 2) * (_psi ** 2) * ((_psi_p1 - _psi_m1) ** 2) / 6 / 12 / _n / _c_s2
    return _tot_t_n

'''
Simulation Part
'''
from pathlib import Path

reproduced_results = Path("reproduced-results")
if not reproduced_results.is_dir():
    reproduced_results.mkdir()
    
def FileNameSimulations(LX, G, tau, forcing, psi_flag):
    return 'LX_' + str(LX) + '_G_' + str(G) + '_tau_' + str(tau) + '_F_' + str(forcing) + '_psi_' + str(psi_flag) +'.hdf5'

_sim_data = defaultdict( # Forcing type
    lambda : defaultdict( # _psi
        lambda : defaultdict( # _G
            lambda : defaultdict (dict) # _tau
        )
    )
)

_st = SimpleTiming()
_st.Start()

_psi_flag = 0
for _forcing in ['guo', 'sc', 'ks']:
    for _tau in [0.8, 1, 1.2]:
        for _psi in psis:
            if _psi == psis[0]:
                _psi_flag = 0
            elif _psi == psis[1]:
                _psi_flag = 1
            for _G in _G_list_sims[_forcing][_tau][_psi]:                    
                _LX, _LY, _psi_code = SimulationsParams(_psi, _forcing, _G, _tau)
                ##_LY = 2 * _LX
                if _LX == False and _LY == False and _psi_code == False:
                    continue
                
                print(_LX, _LY, _psi_code)
                _LX = _LX if _LX % 2 == 1 else _LX + 1
                _direction = 0
                print("LX:", _LX, "G:", _G)
                print()
                print()

                _out_file = reproduced_results / FileNameSimulations(_LX, _G, _tau, _forcing, _psi_flag)

                print()
                print()
                _finished_flag = False
                if not _out_file.is_file():
                    print("File " + str(_out_file) + " not found! Simulating...")

                    sc_eq_cache = ShanChanEquilibriumCache(stencil = _D2E4_P4F4, 
                                                           psi_f = _psi, 
                                                           G = _G, 
                                                           c2 = _c_s2, 
                                                           n_eps = 1e-6)

                    _eq_params = sc_eq_cache.GetFromCache()
                    print("Equilibrium parameters:")
                    print(_eq_params)
                    print()
                    print()

                    _flat_sc = PShanChenMultiPhase(dim_sizes = (_LX, _LY), 
                                                xi_stencil = XIStencils['D2Q9'], 
                                                f_stencil = _D2E4_P4F4.PushStencil(),
                                                psi_code = _psi_code, 
                                                psi_sym = _psi, 
                                                e2_val = 1, 
                                                SC_G = _G, tau = _tau,
                                                lang = _lang, device = _device, 
                                                cl_kind = _kind, 
                                                optimizer_flag = True)


                    _flat_sc.InitFlatInterface(n_g = _eq_params['n_g'], n_l = _eq_params['n_l'], 
                                               width = _LX//2, direction = _direction)

                    if _forcing == 'sc':
                        _flat_sc.MainLoopSCF(range(0, 2**20 + 1, 2**14), 
                                             convergence_functions = [CheckUConvergence])
                    if _forcing == 'guo':
                        _flat_sc.MainLoop(range(0, 2**20 + 1, 2**14), 
                                          convergence_functions = [CheckUConvergence])                
                    if _forcing == 'ks':
                        _flat_sc.MainLoopKSF(range(0, 2**20 + 1, 2**14), 
                                             convergence_functions = [CheckUConvergence])

                    print()
                    print()
                    print("Simulation completed! Dumping in:", _out_file)
                    '''
                    Telling the simulation object to dump density and populations
                    '''
                    _flat_sc.sims_dump_idpy_memory += ['n', 'pop']
                    _flat_sc.DumpSnapshot(file_name = _out_file, custom_types = _flat_sc.custom_types)
                    _finished_flag = True
                    ## Closing simulation
                    _flat_sc.End()

                if _out_file.is_file() or _finished_flag:
                    print("File " + str(_out_file) + " found!")
                    '''
                    Get density
                    '''
                    md = ManageData(dump_file = str(_out_file))
                    _n_swap = md.ReadHDF5(full_key = '/PShanChenMultiPhase/idpy_memory/n', class_check_override=True)

                    # _n_swap = \
                    #     _flat_sc.ReadSnapshotData(
                    #         file_name = _out_file, 
                    #         full_key = _flat_sc.__class__.__name__ + '/idpy_memory/n'
                    #     )

                    _n_swap = _n_swap.reshape(np.flip((_LX, _LY)))
                    _n_swap = _n_swap[0,:]

                    _n_swap_m1 = np.append(_n_swap[-1], _n_swap[:-1])
                    _n_swap_p1 = np.append(_n_swap[1:], _n_swap[0])
                    '''
                    Get Psi
                    '''
                    _psi_f = sp.lambdify(n, _psi)

                    _psi_swap = _psi_f(_n_swap)
                    _psi_swap_m1 = np.append(_psi_swap[-1], _psi_swap[:-1])
                    _psi_swap_p1 = np.append(_psi_swap[1:], _psi_swap[0])

                    if _forcing == 'sc':
                        _p_n_swap = \
                            TotalNormalPSC(_n_swap, _psi_swap, _psi_swap_p1, _psi_swap_m1, 
                                           _G, _tau, _c_s2)
                        
                        _p_t_swap = \
                            TotalTangentialPSC(_n_swap, _psi_swap, _psi_swap_p1, _psi_swap_m1, 
                                               _G, _tau, _c_s2)

                    if _forcing == 'guo':
                        _p_n_swap = \
                            TotalNormalPGuo(_n_swap, _psi_swap, _psi_swap_p1, _psi_swap_m1, 
                                            _G, _tau, _c_s2)
                        _p_t_swap = \
                            TotalTangentialPGuo(_n_swap, _psi_swap, _psi_swap_p1, _psi_swap_m1, 
                                                _G, _tau, _c_s2)

                    if _forcing == 'ks':
                        _p_n_swap = \
                            TotalNormalPKS(_n_swap, _psi_swap, _psi_swap_p1, _psi_swap_m1, 
                                           _G, _tau, _c_s2)
                        _p_t_swap = \
                            TotalTangentialPKS(_n_swap, _psi_swap, _psi_swap_p1, _psi_swap_m1, 
                                               _G, _tau, _c_s2)

                    _p_n_m_t = (_p_n_swap - _p_t_swap)[_LX//2: _LX]
                    _p_n_m_t_spl = \
                        interpolate.UnivariateSpline(np.arange(_LX - _LX//2), _p_n_m_t, k = 4, s = 0)
                    _swap_sigma_0 = _p_n_m_t_spl.integral(0, _LX - _LX//2)
                    _swap_n_l, _swap_n_g, _swap_p_0 = _n_swap[_LX // 2], _n_swap[0], _p_n_swap[0]

                    print(_G)
                    _sim_data[_forcing][_psi][_G][_tau] = \
                        {'n_g': _swap_n_g, 'n_l': _swap_n_l, 
                         'p_0': _swap_p_0, 'sigma_0': _swap_sigma_0, 
                         'p_n': _p_n_swap, 'p_t': _p_t_swap, 'n': _n_swap}
                    
                
    
_sim_obs_list = ['n_g', 'n_l', 'p_0', 'sigma_0', 'p_n', 'p_t', 'n']

_sim_obs_data = defaultdict( # Forcing type
    lambda : defaultdict( # psi
        lambda : defaultdict( # tau
            lambda : defaultdict (dict) # obs
        )
    )
)

print()
for _forcing in ['guo', 'sc', 'ks']:    
    for _tau in [0.8, 1, 1.2]:
        for _psi in psis:
            for _obs in _sim_obs_list:
                _sim_obs_data[_forcing][_psi][_tau][_obs] = []
                _swap_G_list = []
                for _G in _G_list_sims[_forcing][_tau][_psi]:
                    if not SimulationsAnalysisGLimit(_psi, _forcing, _G, _tau):
                        continue

                    _sim_obs_data[_forcing][_psi][_tau][_obs] +=\
                        [_sim_data[_forcing][_psi][_G][_tau][_obs]]
                    _swap_G_list += [_G]

                _sim_obs_data[_forcing][_psi][_tau][_obs] = \
                    np.array(_sim_obs_data[_forcing][_psi][_tau][_obs], dtype=object)

                _sim_obs_data[_forcing][_psi][_tau]['G_list'] = np.array(_swap_G_list)
                
_st.End()
_st.PrintElapsedTime()

## Figure 1

### Panels $(a)$-$(c)$

In [11]:
import matplotlib.pyplot as plt
from matplotlib import rc, rcParams
import matplotlib
from pathlib import Path
from matplotlib.ticker import MaxNLocator


from idpy.Utils.Plots import CreateFiguresPanels, SetDefaultFonts, SetMatplotlibLatexParamas
from idpy.Utils.Plots import SetAxTicksFont, SetAxPanelLabel

reproduced_figures = Path("reproduced-figures")
if not reproduced_figures.is_dir():
    reproduced_figures.mkdir()

SetMatplotlibLatexParamas([rc], [rcParams])

_taus_list = [0.8, 1, 1.2]
_taus_min, _taus_max = min(_taus_list), max(_taus_list)
_cmap_0 = matplotlib.colors.LinearSegmentedColormap.from_list("MyCmapName",["red","blue"])
_cmap_1 = matplotlib.colors.LinearSegmentedColormap.from_list("MyCmapName",["blue","orange"])
_taus_col_norm = matplotlib.colors.LogNorm(vmin = _taus_min, vmax = _taus_max)

def _cmap_0(tau):
    if tau == 0.8:
        return 'blue'
    if tau == 1:
        return 'forestgreen'
    if tau == 1.2:
        return 'orange'

def _linestyle(tau):
    if tau == 0.8:
        return '-'
    if tau == 1:
        return '--'
    if tau == 1.2:
        return ':'
    
def ForcingName(forcing):
    if forcing == 'sc':
        return 'SC'
    if forcing == 'ks':
        return 'Kup'
    if forcing == 'guo':
        return 'Guo'

_fonts = \
    SetDefaultFonts(
        [rc], 
        font_size = 25, legend_font_size = 24, 
        marker_size_small = 7, marker_size_large = 9, 
        marker_size_large_c = 11, 
        thick_line_width = 4, normal_line_width = 2, thin_line_width = 1
    )

'''
Main Text
'''

_nx_fig, _ny_fig = 2, 3
fig = CreateFiguresPanels(_nx = _nx_fig, _ny = _ny_fig, _x_size = 5, _y_size = 2.5)


_panel_label = '(a)'
_ax_coex = plt.subplot2grid((_ny_fig, _nx_fig), (0, 0), colspan = 2, rowspan = 1)

_psi = 0
G_id = 2
_fs_offset = 24
_every = 2
if True:

    '''
    simulations data
    '''

    '''
    LPT
    '''

    _panel_label = '(a)'
    _ax_coex = plt.subplot2grid((_ny_fig, _nx_fig), (0, 0), colspan = 2, rowspan = 1)
    
    '''
    guo for tau = 1.
    '''
    _offset_i = 0
    for _tau in [0.8, 1.0, 1.2]:
        for _forcing in ['guo']:
            data = _sim_obs_data[_forcing][psis[_psi]][_tau]['p_n'][G_id]
            x = np.arange(0,len(data),1)
            
            _ax_coex.plot(
                x, data-np.mean(data),_linestyle(_tau), color=_cmap_0(_tau),linewidth=3,
                label='$\\tau =$'+str(_tau)
            )
            _offset_i += 1

    _ax_coex.set_title("Guo",x=0.95,y=.8,fontsize=26)
    _ax_coex.set_xticks([])
    #_ax_coex.set_ylabel('$P(x)-p_0$', fontsize = _fonts['fs'])
    SetAxTicksFont(_ax_coex, _fonts['fs'])
    _ax_coex.text(3,1.1e-15,'(a)',fontsize=25)
    _ax_coex.text(0.,1.66e-15,'$\\times 10^{-15}$',fontsize=24)
    _ax_coex.get_yaxis().get_offset_text().set_fontsize(_fs_offset)
    _ax_coex.set_xlim([0.,x[-1]])
    _ax_coex.set_ylim([-1.6e-15,1.6e-15])
    _ax_coex.set_yticks([-1e-15,0, 1e-15],['$-1$','$0$','$1$'],fontsize=25)
    # _ax_coex.legend(loc='lower right',bbox_to_anchor=[0.99,-0.05], frameon=False, \
    #                 fancybox=False, shadow=False, ncol=3, fontsize=23)

    _panel_label = '(b)'
    _ax_coex = plt.subplot2grid((_ny_fig, _nx_fig), (1, 0), colspan = 2, rowspan = 1)
    
    '''
    sc for tau = 0.8, 1., 1.2
    '''
    _offset_i = 0
    for _tau in [0.8, 1.0, 1.2]:
        for _forcing in ['sc']:
            data = _sim_obs_data[_forcing][psis[_psi]][_tau]['p_n'][G_id]
            x = np.arange(0,len(data),1)
            
            _ax_coex.plot(
                x, data-np.mean(data),_linestyle(_tau),color=_cmap_0(_tau),linewidth=3,
                label='$\\tau =$'+str(_tau)
            )
            _offset_i += 1

    _ax_coex.set_title("SC",x=0.95,y=.8,fontsize=26)
    _ax_coex.set_xticks([])
    _ax_coex.set_ylabel('$P(x)-p_0$', fontsize = _fonts['fs'])
    SetAxTicksFont(_ax_coex, _fonts['fs'])
    _ax_coex.text(3,1.35e-14,'(b)',fontsize=25)
    _ax_coex.get_yaxis().get_offset_text().set_fontsize(_fs_offset)
    _ax_coex.set_xlim([0.,x[-1]])
    _ax_coex.set_ylim([-1.7e-14,1.95e-14])
    # _ax_coex.legend(loc='lower right',bbox_to_anchor=[0.99,-0.05], frameon=False, \
    #                 fancybox=False, shadow=False, ncol=3, fontsize=23)
    
    _panel_label = '(c)'
    _ax_coex = plt.subplot2grid((_ny_fig, _nx_fig), (2, 0), colspan = 2, rowspan = 1)
    
    '''
    ks for tau = 0.8, 1., 1.2
    '''
    _offset_i = 0
    for _tau in [0.8, 1.0, 1.2]:
        for _forcing in ['ks']:
            data = _sim_obs_data[_forcing][psis[_psi]][_tau]['p_n'][G_id]
            x = np.arange(0,len(data),1)
            #if _tau == 0.8:
            #    x = x*2.1
            _ax_coex.plot(
                x, data-np.mean(data),_linestyle(_tau),color=_cmap_0(_tau),linewidth=3,
                label='$\\tau =$'+str(_tau)
            )
            _offset_i += 1

    _ax_coex.set_title("Kup",x=0.95,y=.8,fontsize=26)
    _ax_coex.set_xticks([])
    _ax_coex.set_xlabel('$x$', fontsize = _fonts['fs'])
    #_ax_coex.set_ylabel('$P(x)-p_0$', fontsize = _fonts['fs'])
    SetAxTicksFont(_ax_coex, _fonts['fs'])
    _ax_coex.text(3,5.72e-14,'(c)',fontsize=25)
    _ax_coex.text(0.,8.25e-14,'$\\times 10^{-14}$',fontsize=24)
    _ax_coex.get_yaxis().get_offset_text().set_fontsize(_fs_offset)
    _ax_coex.set_xlim([0.,x[-1]])
    _ax_coex.set_ylim([-7e-14,8e-14])
    _ax_coex.set_yticks([-6e-14,0, 6e-14],['$-6$','$0$','$6$'],fontsize=25)
    _ax_coex.set_xticks([0.,len(data)/2, len(data)],['$0$','$L/2$','$L$'],fontsize=25)
   # _ax_coex.legend(loc='lower right',bbox_to_anchor=[1.02,-0.08], frameon=False, \
   #                 fancybox=False, shadow=False, handletextpad=0.2,labelspacing=0.,\
   #                columnspacing=0.8,ncol=3, fontsize=23)
     
fig.subplots_adjust(wspace=0, hspace=0)
fig.suptitle('$\\psi_1(n) =\\exp{(-1/n)}$', y=.95,fontsize=26, bbox=dict(facecolor='none', edgecolor='black', pad=6.0))
plt.savefig(reproduced_figures / ('LPT_psi1_closeCritic.pdf'), bbox_inches = 'tight', dpi = 200)
plt.figure()

### Panels $(d)$-$(f)$

In [12]:
import matplotlib.pyplot as plt
from matplotlib import rc, rcParams
import matplotlib
from pathlib import Path


from idpy.Utils.Plots import CreateFiguresPanels, SetDefaultFonts, SetMatplotlibLatexParamas
from idpy.Utils.Plots import SetAxTicksFont, SetAxPanelLabel

reproduced_figures = Path("reproduced-figures")
if not reproduced_figures.is_dir():
    reproduced_figures.mkdir()

SetMatplotlibLatexParamas([rc], [rcParams])

_taus_list = [0.8, 1, 1.2]
_taus_min, _taus_max = min(_taus_list), max(_taus_list)
_cmap_0 = matplotlib.colors.LinearSegmentedColormap.from_list("MyCmapName",["red","blue"])
_cmap_1 = matplotlib.colors.LinearSegmentedColormap.from_list("MyCmapName",["blue","orange"])
_taus_col_norm = matplotlib.colors.LogNorm(vmin = _taus_min, vmax = _taus_max)

def _cmap_0(tau):
    if tau == 0.8:
        return 'blue'
    if tau == 1:
        return 'forestgreen'
    if tau == 1.2:
        return 'orange'

def _linestyle(tau):
    if tau == 0.8:
        return '-'
    if tau == 1:
        return '--'
    if tau == 1.2:
        return ':'
    
def ForcingName(forcing):
    if forcing == 'sc':
        return 'SC'
    if forcing == 'ks':
        return 'Kup'
    if forcing == 'guo':
        return 'Guo'

_fonts = \
    SetDefaultFonts(
        [rc], 
        font_size = 25, legend_font_size = 24, 
        marker_size_small = 7, marker_size_large = 9, 
        marker_size_large_c = 11, 
        thick_line_width = 4, normal_line_width = 2, thin_line_width = 1
    )

'''
Main Text
'''

_nx_fig, _ny_fig = 2, 3
fig = CreateFiguresPanels(_nx = _nx_fig, _ny = _ny_fig, _x_size = 5, _y_size = 2.5)


_panel_label = '(a)'
_ax_coex = plt.subplot2grid((_ny_fig, _nx_fig), (0, 0), colspan = 2, rowspan = 1)

_psi = 1
G_id = 2
_fs_offset = 24
_every = 2
if True:

    '''
    simulations data
    '''

    '''
    LPT
    '''

    _panel_label = '(a)'
    _ax_coex = plt.subplot2grid((_ny_fig, _nx_fig), (0, 0), colspan = 2, rowspan = 1)
    
    '''
    guo for tau = 1.
    '''
    _offset_i = 0
    for _tau in [0.8, 1.0, 1.2]:
        for _forcing in ['guo']:
            data = _sim_obs_data[_forcing][psis[_psi]][_tau]['p_n'][G_id]
            x = np.arange(0,len(data),1)
            
            _ax_coex.plot(
                x, data-np.mean(data),_linestyle(_tau), color=_cmap_0(_tau),linewidth=3,
                label='$\\tau =$'+str(_tau)
            )
            _offset_i += 1

    _ax_coex.set_title("Guo",x=0.95,y=.8,fontsize=26)
    _ax_coex.set_xticks([])
    #_ax_coex.set_ylabel('$P(x)-p_0$', fontsize = _fonts['fs'])
    SetAxTicksFont(_ax_coex, _fonts['fs'])
    #_ax_coex.text(3,3e-14,'(d)',fontsize=25)
    _ax_coex.text(3,2.5e-14,'(d)',fontsize=25)
    # _ax_coex.get_yaxis().get_offset_text().set_fontsize(_fs_offset)
    _ax_coex.set_xlim([0.,x[-1]])
    _ax_coex.set_ylim([-3.5e-14,3.65e-14])
    # _ax_coex.set_yticks([-1e-13,0, 1e-13],['$-1$','$0$','$1$'],fontsize=25)
    # _ax_coex.legend(loc='lower right',bbox_to_anchor=[0.99,-0.05], frameon=False, \
    #                 fancybox=False, shadow=False, ncol=3, fontsize=23)

    _panel_label = '(b)'
    _ax_coex = plt.subplot2grid((_ny_fig, _nx_fig), (1, 0), colspan = 2, rowspan = 1)
    
    '''
    sc for tau = 0.8, 1., 1.2
    '''
    _offset_i = 0
    for _tau in [0.8, 1.0, 1.2]:
        for _forcing in ['sc']:
            data = _sim_obs_data[_forcing][psis[_psi]][_tau]['p_n'][G_id]
            x = np.arange(0,len(data),1)
            
            _ax_coex.plot(
                x, data-np.mean(data),_linestyle(_tau),color=_cmap_0(_tau),linewidth=3,
                label='$\\tau =$'+str(_tau)
            )
            _offset_i += 1

    _ax_coex.set_title("SC",x=0.95,y=.8,fontsize=26)
    _ax_coex.set_xticks([])
    _ax_coex.set_ylabel('$P(x)-p_0$', fontsize = _fonts['fs'])
    SetAxTicksFont(_ax_coex, _fonts['fs'])
    _ax_coex.text(3,3.35e-14,'(e)',fontsize=25)
    _ax_coex.text(0.,5.25e-14,'$\\times 10^{-14}$',fontsize=24)
    _ax_coex.get_yaxis().get_offset_text().set_fontsize(_fs_offset)
    _ax_coex.set_xlim([0.,x[-1]])
    _ax_coex.set_ylim([-5e-14,5e-14])
    _ax_coex.set_yticks([-4e-14,0, 4e-14],['$-4$','$0$','$4$'],fontsize=25)
    #_ax_coex.set_ylim([-2e-12,2.5e-12])
    # _ax_coex.legend(loc='lower right',bbox_to_anchor=[0.99,-0.05], frameon=False, \
    #                 fancybox=False, shadow=False, ncol=3, fontsize=23)
    
    _panel_label = '(c)'
    _ax_coex = plt.subplot2grid((_ny_fig, _nx_fig), (2, 0), colspan = 2, rowspan = 1)
    
    '''
    ks for tau = 0.8, 1., 1.2
    '''
    _offset_i = 0
    for _tau in [0.8, 1.0, 1.2]:
        for _forcing in ['ks']:
            data = _sim_obs_data[_forcing][psis[_psi]][_tau]['p_n'][G_id]
            x = np.arange(0,len(data),1)
            #if _tau == 0.8:
            #    x = x*2.1
            _ax_coex.plot(
                x, data-np.mean(data),_linestyle(_tau),color=_cmap_0(_tau),linewidth=3,
                label='$\\tau =$'+str(_tau)
            )
            _offset_i += 1

    _ax_coex.set_title("Kup",x=0.95,y=.8,fontsize=26)
    _ax_coex.set_xticks([])
    _ax_coex.set_xlabel('$x$', fontsize = _fonts['fs'])
    #_ax_coex.set_ylabel('$P(x)-p_0$', fontsize = _fonts['fs'])
    SetAxTicksFont(_ax_coex, _fonts['fs'])
    _ax_coex.text(3,4.1e-14,'(f)',fontsize=25)
    _ax_coex.text(0.,6.25e-14,'$\\times 10^{-14}$',fontsize=24)
    _ax_coex.get_yaxis().get_offset_text().set_fontsize(_fs_offset)
    _ax_coex.set_xlim([0.,x[-1]])
    _ax_coex.set_ylim([-5.5e-14,6e-14])
    _ax_coex.set_yticks([-4e-14,0, 4e-14],['$-4$','$0$','$4$'],fontsize=25)
    _ax_coex.set_xticks([0.,len(data)/2, len(data)],['$0$','$L/2$','$L$'],fontsize=25)
     
fig.subplots_adjust(wspace=0, hspace=0)
fig.suptitle('$\\psi_2(n) = 1-\\exp{(-n)}$', y=.95,fontsize=26,bbox=dict(facecolor='none', edgecolor='black', pad=6.0))
plt.savefig(reproduced_figures / ('LPT_psi2_closeCritic.pdf'), bbox_inches = 'tight', dpi = 200)
plt.figure()

## Figure 2

In [13]:
import matplotlib.pyplot as plt
from matplotlib import rc, rcParams
import matplotlib
from pathlib import Path


from idpy.Utils.Plots import CreateFiguresPanels, SetDefaultFonts, SetMatplotlibLatexParamas
from idpy.Utils.Plots import SetAxTicksFont, SetAxPanelLabel

reproduced_figures = Path("reproduced-figures")
if not reproduced_figures.is_dir():
    reproduced_figures.mkdir()

SetMatplotlibLatexParamas([rc], [rcParams])

_taus_list = [0.8, 1, 1.2]
_taus_min, _taus_max = min(_taus_list), max(_taus_list)
_cmap_0 = matplotlib.colors.LinearSegmentedColormap.from_list("MyCmapName",["red","blue"])
_cmap_1 = matplotlib.colors.LinearSegmentedColormap.from_list("MyCmapName",["blue","orange"])
_taus_col_norm = matplotlib.colors.LogNorm(vmin = _taus_min, vmax = _taus_max)

def _cmap_0(tau):
    if tau == 0.8:
        return 'blue'
    if tau == 1:
        return 'forestgreen'
    if tau == 1.2:
        return 'orange'
    
def ForcingName(forcing):
    if forcing == 'sc':
        return 'SC LPT'
    if forcing == 'guo':
        return 'Guo LPT'

_fonts = \
    SetDefaultFonts(
        [rc], 
        font_size = 18, legend_font_size = 17, 
        marker_size_small = 7, marker_size_large = 9, 
        marker_size_large_c = 11, 
        thick_line_width = 4, normal_line_width = 2, thin_line_width = 1
    )


'''
Here you decide the colors
'''
_symbols_style = \
    {'guo': {0.8: {'marker': 'o', 'linestyle': 'none', 'fillstyle': 'none', 
                   'markersize': _fonts['marker_size_large'], 'color': _cmap_0((0.8))}, 
             1: {'marker': 'o', 'linestyle': 'none', 'fillstyle': 'none', 
                 'markersize': _fonts['marker_size_large'], 'color': _cmap_0((1))}, 
             1.2: {'marker': 'o', 'linestyle': 'none', 'fillstyle': 'none', 
                   'markersize': _fonts['marker_size_large'], 'color': _cmap_0((1.2))}}, 
     
     'sc': {0.8: {'marker': '^', 'linestyle': 'none', 'fillstyle': 'none', 
                   'markersize': _fonts['marker_size_large'], 'color': _cmap_0((0.8))}, 
             1: {'marker': '^', 'linestyle': 'none', 'fillstyle': 'none', 
                 'markersize': _fonts['marker_size_large'], 'color': _cmap_0((1))}, 
             1.2: {'marker': '^', 'linestyle': 'none', 'fillstyle': 'none', 
                   'markersize': _fonts['marker_size_large'], 'color': _cmap_0((1.2))}}, 
    
     'ks': {0.8: {'marker': 's', 'linestyle': 'none', 'fillstyle': 'none', 
                   'markersize': _fonts['marker_size_large'], 'color': _cmap_0((0.8))}, 
             1: {'marker': 's', 'linestyle': 'none', 'fillstyle': 'none', 
                 'markersize': _fonts['marker_size_large'], 'color': _cmap_0((1))}, 
             1.2: {'marker': 's', 'linestyle': 'none', 'fillstyle': 'none', 
                   'markersize': _fonts['marker_size_large'], 'color': _cmap_0((1.2))}}     
    }

'''
Here you decide symbol position in the table
'''
_positionX = \
        {0.8: 0.24,
         1: 0.35,
         1.2: 0.54
    }
_positionY = \
    {'mod': 0.52,
     'guo': 0.49,
     'sc' : 0.46,
     'ks' : 0.43    
    }
'''
Main Text
'''

_nx_fig, _ny_fig = 2, 3
fig = CreateFiguresPanels(_nx = _nx_fig, _ny = _ny_fig, _x_size = 6.5, _y_size = 5.)


_panel_label = '(a)'
_ax_coex = plt.subplot2grid((_ny_fig, _nx_fig), (0, 0), colspan = 2, rowspan = 1)

_ax_coex.axvspan(0.1, 1., facecolor='skyblue', alpha=0.2)
_ax_coex.axvspan(1., 40, facecolor='lightgrey', alpha=0.2)
_ax_coex.text(0.15, 0.825, "Liquid", size=15)
_ax_coex.text(0.16, 0.78, "$(n_l)$", size=15)
_ax_coex.text(15,0.825, 'Gas', size=15)
_ax_coex.text(15.3, 0.78, "$(n_g)$", size=15)
_ax_coex.text(0.14, 0.96,'$\\psi_1(n) =\\exp{(-1/n)}$', size=15,
        bbox=dict(facecolor='none', edgecolor='black', pad=6.0))

if True:
    '''
    Theory lines
    '''
    for _forcing in ['guo', 'sc']:
        for _psi in psis[:1]:
            _swap_taus = [1] if _forcing == 'guo' else _taus_list
            for _tau in _swap_taus:
                _color = _cmap_0((_tau)) if _forcing == 'sc' else 'black'
                _width = _fonts['thin_line_width'] if _forcing != 'guo' else _fonts['thin_line_width'] * 1.
                _lstyle = '-.' if _forcing != 'guo' else '-'
                
                _ax_coex.plot(
                        _nc[_psi]/_th_obs_data[_forcing][_psi][_tau]['n_g'], 
                        _th_obs_data[_forcing][_psi][_tau]['G_c'] / \
                        _th_obs_data[_forcing][_psi][_tau]['G_list'], 
                        color = _color, linestyle = _lstyle, linewidth = _width
        #                label = str(ForcingName(_forcing)) +', $\\tau = ' + str(_tau) + '$'
                )
    #plt.legend(loc=(0.75,0.2), frameon=False, fontsize=15)
    
    for _forcing in ['guo']:
        for _psi in psis[:1]:
            for _tau in [1]:
                _ax_coex.plot(
                        _nc[_psi]/_th_obs_data[_forcing][_psi][_tau]['n_l'], 
                        _th_obs_data[_forcing][_psi][_tau]['G_c'] / \
                        _th_obs_data[_forcing][_psi][_tau]['G_list'], 
                        color = 'black', linestyle = '-', linewidth = _fonts['thin_line_width']
                )

    '''
    simulations data
    '''
    for _psi in psis[:1]:
        '''
        Gas densities first
        '''
        '''
        sc for tau = 0.8 and 1.2
        '''
        _offset_i = 0
        for _tau in [0.8, 1.2]:
            for _forcing in ['sc']:
                _ax_coex.plot(
                    _nc[_psi]/_sim_obs_data[_forcing][_psi][_tau]['n_g'][::2][:-2], 
                    _Gc[_psi]/_sim_obs_data[_forcing][_psi][_tau]['G_list'][::2][:-2], 
                    **_symbols_style[_forcing][_tau], markeredgewidth=1
                )
                _offset_i += 1
        
        '''
        mod + guo: superpose: three taus: one point every 6
        '''
        _offset_i = 0
        for _tau in [0.8,1.2]:
            for _forcing in ['guo','ks']: #'mod', 'guo','ks'
                _ax_coex.plot(
                    _nc[_psi]/_sim_obs_data[_forcing][_psi][_tau]['n_g'][_offset_i::3], #every 6
                    _Gc[_psi]/_sim_obs_data[_forcing][_psi][_tau]['G_list'][_offset_i::3], #every 6
                    **_symbols_style[_forcing][_tau], markeredgewidth=1
                )
                _offset_i += 1

        '''
        mod + guo: superpose: three taus: one point every 6
        '''
        _offset_i = 0
        for _tau in [1]:
            for _forcing in ['sc']: #'sc', 'mod'
                _ax_coex.plot(
                    _nc[_psi]/_sim_obs_data[_forcing][_psi][_tau]['n_g'][_offset_i::6], 
                    _Gc[_psi]/_sim_obs_data[_forcing][_psi][_tau]['G_list'][_offset_i::6], 
                    **_symbols_style[_forcing][_tau],markeredgewidth=1
                )
                _offset_i += 1
        
        _offset_i = 0
        for _tau in [1]:                
            for _forcing in ['ks', 'guo']:
                _ax_coex.plot(
                    _nc[_psi]/_sim_obs_data[_forcing][_psi][_tau]['n_g'][_offset_i::3], #every 4
                    _Gc[_psi]/_sim_obs_data[_forcing][_psi][_tau]['G_list'][_offset_i::3], #every 4
                    **_symbols_style[_forcing][_tau], markeredgewidth=1
                )
                _offset_i += 1
                
                
        '''
        Liquid densities
        '''
        _offset_i = 0
        for _tau in _taus_list:
            for _forcing in ['guo', 'sc', 'ks']: #'mod', 'guo', 'sc', 'ks'
                _ax_coex.plot(
                    _nc[_psi]/_sim_obs_data[_forcing][_psi][_tau]['n_l'][_offset_i::9], #every 12
                    _Gc[_psi]/_sim_obs_data[_forcing][_psi][_tau]['G_list'][_offset_i::9], #every 12
                    **_symbols_style[_forcing][_tau],markeredgewidth=1
                )
                _offset_i += 1

    _ax_coex.set_xlabel('$n_c / n$', fontsize = _fonts['fs'])
    _ax_coex.set_ylabel('$G_c / G$', fontsize = _fonts['fs'])
    SetAxTicksFont(_ax_coex, _fonts['fs'])
    SetAxPanelLabel(_ax_coex, _panel_label, pos = 'ul', fs = 18)


    _ax_coex.set_xscale('log')
    _ax_coex.set_yscale('log')
    _ax_coex.set_xlim([0.1,22])
    _ax_coex.set_ylim([0.38,1.05])
    _ax_coex.set_xticks([])
    _ax_coex.set_yticks([0.4,0.6, 1],['$4 \cdot 10^{-1}$','$6 \cdot 10^{-1}$','$10^{0}$'])
    


_panel_label = '(b)'
_ax_coex = plt.subplot2grid((_ny_fig, _nx_fig), (1, 0), colspan = 2, rowspan = 1)

_ax_coex.axvspan(0.1, 1., facecolor='skyblue', alpha=0.2)
_ax_coex.axvspan(1., 40, facecolor='lightgrey', alpha=0.2)
_ax_coex.text(0.15, 0.825, "Liquid", size=15)
_ax_coex.text(0.16, 0.78, "$(n_l)$", size=15)
_ax_coex.text(15,0.825, 'Gas', size=15)
_ax_coex.text(15.3, 0.78, "$(n_g)$", size=15)
_ax_coex.text(0.14, 0.96,'$\\psi_2(n) =1-\\exp{(-n)}$', size=15,
        bbox=dict(facecolor='none', edgecolor='black', pad=6.0))

if True:
    
    '''
    Theory lines
    '''
    for _forcing in ['guo', 'sc']:
        for _psi in psis[1:]:
            _swap_taus = [1] if _forcing == 'guo' else _taus_list
            for _tau in _swap_taus:
                _color = _cmap_0((_tau)) if _forcing == 'sc' else 'black'
                _width = _fonts['thin_line_width'] if _forcing != 'guo' else _fonts['thin_line_width'] * 1.
                _lstyle = '-.' if _forcing != 'guo' else '-'
                
                _ax_coex.plot(_nc[_psi]/_th_obs_data[_forcing][_psi][_tau]['n_g'], 
                        _th_obs_data[_forcing][_psi][_tau]['G_c'] / \
                              _th_obs_data[_forcing][_psi][_tau]['G_list'], 
                        color = _color, linestyle = _lstyle, linewidth = _width,
                              label = str(ForcingName(_forcing)) +', $\\tau = ' + str(_tau) + '$'
                )
    plt.legend(loc=(0.45,0.04), frameon=False, fontsize=14)
    
    for _forcing in ['guo']:
        for _psi in psis[1:]:
            for _tau in [1]:
                _ax_coex.plot(_nc[_psi]/_th_obs_data[_forcing][_psi][_tau]['n_l'], 
                        _th_obs_data[_forcing][_psi][_tau]['G_c'] / \
                              _th_obs_data[_forcing][_psi][_tau]['G_list'], 
                    color = 'black', linestyle = '-', linewidth = _fonts['thin_line_width']
                )
    
    '''
    simulations data
    '''
    for _psi in psis[1:]:
        '''
        Gas densities first
        '''
        '''
        sc for tau = 0.8 and 1.2
        '''
        _offset_i = 0
        for _tau in [0.8, 1.2]:
            for _forcing in ['sc']:
                _ax_coex.plot(
                    _nc[_psi]/_sim_obs_data[_forcing][_psi][_tau]['n_g'][::1],  #every 2
                    _Gc[_psi]/_sim_obs_data[_forcing][_psi][_tau]['G_list'][::1],  #every 2
                    **_symbols_style[_forcing][_tau], markeredgewidth=1
                )
                _offset_i += 1

        
        '''
        mod + guo: superpose: three taus: one point every 6
        '''
        _offset_i = 0
        for _tau in _taus_list:
            for _forcing in ['guo']: #'mod', 'guo'
                _ax_coex.plot(
                    _nc[_psi]/_sim_obs_data[_forcing][_psi][_tau]['n_g'][_offset_i::3], #every 6
                    _Gc[_psi]/_sim_obs_data[_forcing][_psi][_tau]['G_list'][_offset_i::3], #every 6
                    **_symbols_style[_forcing][_tau], markeredgewidth=1
                )
                _offset_i += 1


        '''
        mod + guo: superpose: three taus: one point every 6
        '''
        _offset_i = 0
        for _tau in [1]:
            for _forcing in ['sc']:
                _ax_coex.plot(
                    _nc[_psi]/_sim_obs_data[_forcing][_psi][_tau]['n_g'][_offset_i::6],   #every 6
                    _Gc[_psi]/_sim_obs_data[_forcing][_psi][_tau]['G_list'][_offset_i::6],  #every 6
                    **_symbols_style[_forcing][_tau],markeredgewidth=1
                )
                _offset_i += 1

                
        for _tau in [1]:                
            for _forcing in ['ks', 'guo']:
                _ax_coex.plot(
                    _nc[_psi]/_sim_obs_data[_forcing][_psi][_tau]['n_g'][_offset_i::3],    #every 4
                    _Gc[_psi]/_sim_obs_data[_forcing][_psi][_tau]['G_list'][_offset_i::3], #every 4
                    **_symbols_style[_forcing][_tau], markeredgewidth=1
                )
                _offset_i += 1
                
                
        '''
        Liquid densities
        '''
        _offset_i = 0
        for _tau in _taus_list:
            for _forcing in ['guo', 'sc', 'ks']: #'mod', 'guo', 'sc', 'ks'
                _ax_coex.plot(
                    _nc[_psi]/_sim_obs_data[_forcing][_psi][_tau]['n_l'][_offset_i::9],    #every 12
                    _Gc[_psi]/_sim_obs_data[_forcing][_psi][_tau]['G_list'][_offset_i::9], #every 12
                    **_symbols_style[_forcing][_tau],markeredgewidth=1
                )
                _offset_i += 1
                plt.plot(_positionX[_tau],_positionY[_forcing], **_symbols_style[_forcing][_tau])
    
    
    _ax_coex.set_xlabel('$n_c / n$', fontsize = _fonts['fs'])
    _ax_coex.set_ylabel('$G_c / G$', fontsize = _fonts['fs'])
    SetAxTicksFont(_ax_coex, _fonts['fs'])
    SetAxPanelLabel(_ax_coex, _panel_label, pos = 'ul', fs = 18)
    
    col_labels=['$\\tau = 0.8$','$\\tau = 1.0$','$\\\tau = 1.2$']
    row_labels=['Guo','SC','Kup'] #'Mod. SC','Guo','SC','Kup.'
    table_vals=[11,12,13,21,22,23,31,32,33,31,32,33]

    table = r'''\begin{tabular}{ c | c  c  c } & $\tau = 0.8$ & $\tau = 1.0$ & $\tau = 1.2$ \\\hline Guo & & & \\ SC & & & \\ Kup & & & \end{tabular}'''
    #plt.text(0.12,0.48,table,size=15) 
    plt.text(0.15,0.467,table,size=14) 
    
    _ax_coex.set_xscale('log')
    _ax_coex.set_yscale('log')
    _ax_coex.set_xlim([0.1,22])
    _ax_coex.set_ylim([0.38,1.05])
    _ax_coex.set_yticks([0.4,0.6, 1],['$4 \cdot 10^{-1}$','$6 \cdot 10^{-1}$','$10^{0}$'])

fig.subplots_adjust(wspace=0, hspace=0)
plt.savefig(reproduced_figures / ('coexistence.pdf'), bbox_inches = 'tight', dpi = 200)
plt.figure()